# SAM 3 + Qwen — AI Artwork Extraction Inference Server (Colab T4)

Notebook này triển khai backend FastAPI inference server trên Google Colab T4 16GB, tích hợp Meta SAM 3 image model và Qwen3-VL-4B Prompt Assistant (4-bit), đồng thời mở Cloudflare Quick Tunnel để kết nối với React Web UI trên máy designer.

## Trước khi chạy

1. **Mở Runtime → Change runtime type**.
2. Chọn **Hardware accelerator: T4 GPU**.
3. Ở **Runtime Version**, chọn **2026.07 (Python 3.12.13)** để tránh xung đột dependency PyTorch/CUDA.
4. Đăng nhập Hugging Face và được cấp quyền truy cập gated model: [facebook/sam3](https://huggingface.co/facebook/sam3).
5. Chuẩn bị Hugging Face **Read** token (lưu trong Colab Secrets với tên `HF_TOKEN` hoặc nhập trực tiếp).

## Cảnh báo quan trọng

- **Môi trường tạm thời (Ephemeral Runtime)**: Dữ liệu trên `/content` sẽ bị xóa khi session bị ngắt.
- **Quick Tunnel Public URL**: URL `trycloudflare.com` là public qua internet. Server bắt buộc xác thực qua Bearer Token ngẫu nhiên được sinh trong notebook.
- **Bảo mật dữ liệu**: Không upload hình ảnh chứa dữ liệu nhạy cảm hoặc bí mật kinh doanh lên endpoint test khi chưa được cho phép.


## 1. Kiểm tra môi trường runtime và GPU

Chạy cell này trước. Nếu không thấy GPU NVIDIA hoặc Python không phải 3.12, hãy kiểm tra lại Hardware accelerator.

In [ ]:
import sys, torch, platform

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if sys.version_info[:2] != (3, 12):
    print("\nWARNING: Notebook này được chuẩn bị để chạy với Colab Runtime Version 2026.07 (Python 3.12.13).")

if not torch.cuda.is_available():
    raise RuntimeError(
        "Chưa có GPU. Vào Runtime → Change runtime type → Hardware accelerator → T4 GPU, rồi chạy lại cell."
    )

props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")

if props.total_memory < 12 * 1024**3:
    print("WARNING: GPU có ít hơn 12GB VRAM; inference có thể gặp CUDA out-of-memory.")


## 1b. Cấu hình chung cho toàn bộ notebook

Tất cả các cờ cấu hình runtime được thiết lập duy nhất tại cell này. Nếu gặp lỗi OOM hoặc muốn chạy chế độ nhẹ nhất (chỉ SAM 3), đặt `ENABLE_QWEN = False` tại đây rồi chạy lần lượt các cell tiếp theo.

In [ ]:
# ==============================================================================
# CẤU HÌNH RUNTIME CHUNG (Chỉnh sửa tại đây và không gán lại ở các cell sau)
# ==============================================================================
ENABLE_QWEN = True               # Bật/tắt Qwen Prompt Assistant
QWEN_MIN_FREE_VRAM_GB = 5.0     # Ngưỡng VRAM trống tối thiểu để nạp Qwen (GB)
USE_DRIVE_CACHE = False          # Lưu checkpoint Hugging Face vào Google Drive
MAX_UPLOAD_MB = 20               # Giới hạn dung lượng file ảnh upload (MB)
DEFAULT_THRESHOLD = 0.35         # Ngưỡng confidence threshold mặc định cho SAM 3
MAX_CANDIDATES = 12              # Số lượng candidate tối đa xuất ra trong file ZIP
OVERLAY_MAX_SIDE = 768           # Kích thước cạnh tối đa của overlay preview (lean profile)
CORS_ORIGINS = ["http://localhost:5173", "http://127.0.0.1:5173"]  # Origins cho phép kết nối Web UI
SAM_DEVICE = "cuda"              # Thiết bị chạy SAM 3 ("cuda" hoặc "cpu")

print("Cấu hình runtime đã được thiết lập:")
print(f"  ENABLE_QWEN           : {ENABLE_QWEN}")
print(f"  QWEN_MIN_FREE_VRAM_GB : {QWEN_MIN_FREE_VRAM_GB} GB")
print(f"  USE_DRIVE_CACHE       : {USE_DRIVE_CACHE}")
print(f"  MAX_UPLOAD_MB         : {MAX_UPLOAD_MB} MB")
print(f"  DEFAULT_THRESHOLD     : {DEFAULT_THRESHOLD}")
print(f"  MAX_CANDIDATES        : {MAX_CANDIDATES}")
print(f"  OVERLAY_MAX_SIDE      : {OVERLAY_MAX_SIDE}")
print(f"  CORS_ORIGINS          : {CORS_ORIGINS}")
print(f"  SAM_DEVICE            : {SAM_DEVICE}")


## 2. Mount Google Drive tùy chọn để cache checkpoint Hugging Face

Sử dụng cờ `USE_DRIVE_CACHE` đã cấu hình ở Mục 1b. Nếu bật, thư mục cache Hugging Face sẽ được đặt trên Google Drive để tránh tải lại checkpoint mỗi lần khởi động.

In [ ]:
import os

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_hf_cache = "/content/drive/MyDrive/huggingface_cache"
    os.makedirs(drive_hf_cache, exist_ok=True)
    os.environ["HF_HOME"] = drive_hf_cache
    print(f"HF_HOME set to Google Drive: {drive_hf_cache}")
else:
    print("Drive cache disabled (USE_DRIVE_CACHE = False). Using default Colab local storage.")


## 3. Cài đặt repository SAM 3 và server dependencies

Cài package từ repository facebookresearch/sam3 chính thức và các thư viện FastAPI, Uvicorn, Multipart. Dependencies của Qwen chỉ được cài nếu `ENABLE_QWEN = True`. Kiểm tra phiên bản PyTorch trước và sau khi cài đặt thông qua tiến trình con độc lập để đảm bảo không bị thay đổi.

In [ ]:
import os, subprocess, sys, torch

orig_torch_version = torch.__version__
print(f"Pre-install PyTorch version: {orig_torch_version}")

# Clone vào sam3_repo (KHÔNG đặt tên /content/sam3): nếu thư mục repo trùng tên package,
# `import sam3` từ /content sẽ trỏ nhầm vào thư mục repo -> No module named 'sam3.model_builder'.
repo_dir = "/content/sam3_repo"
if not os.path.isdir(repo_dir):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/facebookresearch/sam3.git", repo_dir],
        check=True,
    )

%cd /content/sam3_repo

# Cài SAM 3 theo pyproject.toml của repository; không cài lại torch/torchvision.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "fastapi", "uvicorn", "python-multipart", "pillow", "opencv-python-headless", "matplotlib",
    ],
    check=True,
)

# Cài đặt Qwen dependencies có kiểm tra điều kiện (không nâng cấp torch)
if ENABLE_QWEN:
    print(f"Installing Qwen dependencies (transformers>=4.57, accelerate, bitsandbytes) with PyTorch {orig_torch_version}...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "transformers>=4.57", "accelerate", "bitsandbytes",
        ],
        check=True,
    )
else:
    print("ENABLE_QWEN is False: skipping Qwen dependencies installation.")

%cd /content

# `pip install -e .` chỉ có hiệu lực sau khi khởi động lại Python, nên thêm repo vào sys.path
# cho kernel hiện tại và bỏ mọi bản `sam3` đã import nhầm trước đó.
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
for _mod in [m for m in sys.modules if m == "sam3" or m.startswith("sam3.")]:
    del sys.modules[_mod]
import sam3.model_builder  # fail sớm nếu SAM 3 chưa cài đúng
import sam3
print("sam3 OK:", sam3.__file__)

# Kiểm tra PyTorch version không bị thay đổi bởi pip thông qua tiến trình con độc lập
check_proc = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True,
    text=True,
)
post_torch_version = check_proc.stdout.strip()
print(f"Post-install PyTorch version: {post_torch_version}")
if post_torch_version != orig_torch_version:
    print(f"\nCRITICAL WARNING: PyTorch version changed on disk from {orig_torch_version} to {post_torch_version}!")
    print("This may break CUDA support. Please restart session via Runtime -> Restart session.\n")
else:
    print("PyTorch version verified unchanged on disk. Compatible with Colab GPU runtime.")


## 4. Đăng nhập Hugging Face

Checkpoint SAM 3 nằm trong repository gated `facebook/sam3`. Bạn cần token loại **Read**. Token được lấy tự động từ Colab Secret `HF_TOKEN` nếu có, hoặc nhập ẩn qua prompt mà không in ra màn hình.

In [ ]:
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    from getpass import getpass
    hf_token = getpass("Paste Hugging Face READ token (input hidden): ").strip()

if not hf_token:
    raise ValueError("Bạn chưa nhập Hugging Face token.")

login(token=hf_token, add_to_git_credential=False)
del hf_token
print("Hugging Face login completed successfully.")


## 5. Tạo server source code trong runtime Colab

Ghi toàn bộ module từ `server_reference/` vào `/content/server_reference/` và `colab/server_main.py` vào `/content/server_main.py`, sau đó thêm `/content` vào `sys.path` để có thể import trực tiếp.

In [ ]:
import os
os.makedirs("/content/server_reference", exist_ok=True)
print("Created /content/server_reference directory.")


In [ ]:
%%writefile /content/server_reference/__init__.py


In [ ]:
%%writefile /content/server_reference/api.py
"""FastAPI application for AI Artwork Extractor reference server.

Implements contract §1 and project plan §7.
Endpoints:
  GET /health
  POST /api/v1/assistant/prompt
  POST /api/v1/segment
"""
from __future__ import annotations

import io
import json
import logging
import secrets
import threading
import time
from typing import TYPE_CHECKING, Any, Optional

from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, Request, Response, UploadFile
from fastapi.exceptions import RequestValidationError
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from PIL import Image, ImageOps

from server_reference.errors import PromptAssistantOutputError, PromptAssistantUnavailable
from server_reference.image_export import build_result_zip
from server_reference.sam3_engine import gpu_name
from server_reference.schemas import HealthResponse, Mode, PromptSuggestion, ScopePreference

if TYPE_CHECKING:
    from server_reference.sam3_engine import Sam3Engine

logger = logging.getLogger("server_reference.api")
ALLOWED_FORMATS = ("JPEG", "JPG", "PNG", "WEBP")


def _read_and_validate_image(file: UploadFile, max_upload_mb: int) -> Image.Image:
    """Read uploaded file with size cap and validate format.

    Raises:
        HTTPException 413 if upload exceeds max_upload_mb.
        HTTPException 415 if file is not a supported decodable image.
    """
    max_bytes = max_upload_mb * 1024 * 1024
    content = file.file.read(max_bytes + 1)
    if len(content) > max_bytes:
        raise HTTPException(status_code=413, detail=f"Upload exceeds maximum allowed size of {max_upload_mb}MB")

    try:
        img_probe = Image.open(io.BytesIO(content))
        img_probe.verify()
    except Exception:
        raise HTTPException(status_code=415, detail="Uploaded file is not a valid decodable image")

    fmt = (img_probe.format or "").upper()
    if fmt not in ALLOWED_FORMATS:
        raise HTTPException(status_code=415, detail=f"Image format '{fmt}' is not supported. Use JPG, PNG, or WebP.")

    # Re-open clean image after verify() and apply orientation
    img = Image.open(io.BytesIO(content))
    img = ImageOps.exif_transpose(img)
    return img


def create_app(
    sam_engine: Optional[Any] = None,
    assistant: Optional[Any] = None,
    api_token: str = "",
    max_upload_mb: int = 20,
    default_threshold: float = 0.35,
    max_candidates: int = 12,
    cors_origins: Optional[list[str]] = None,
    overlay_max_side: int = 768,
) -> FastAPI:
    """Create and configure the FastAPI application."""
    if not api_token or not api_token.strip() or len(api_token.strip()) < 16:
        raise ValueError("api_token must be a non-empty string of at least 16 characters")
    api_token = api_token.strip()

    app = FastAPI(title="AI Artwork Extractor Colab Server")
    lock = threading.Lock()
    app.state.lock = lock

    # Configure CORS for browser Web UI per contract §1
    allowed_origins = (
        list(cors_origins)
        if cors_origins is not None
        else ["http://localhost:5173", "http://127.0.0.1:5173"]
    )
    app.add_middleware(
        CORSMiddleware,
        allow_origins=allowed_origins,
        allow_credentials=False,
        allow_methods=["GET", "POST", "OPTIONS"],
        allow_headers=["Authorization", "Content-Type"],
        expose_headers=["Content-Disposition"],
    )

    @app.exception_handler(RequestValidationError)
    def validation_exception_handler(_: Request, exc: RequestValidationError) -> JSONResponse:
        first_err = exc.errors()[0] if exc.errors() else {}
        msg = first_err.get("msg", "Validation error")
        loc = ".".join(str(x) for x in first_err.get("loc", []) if x != "body")
        detail = f"{loc}: {msg}" if loc else msg
        return JSONResponse(status_code=422, content={"detail": detail})

    def verify_auth(authorization: Optional[str] = Header(None)) -> None:
        """Enforce Bearer token authorization header case-insensitively per RFC 6750."""
        if not authorization:
            raise HTTPException(
                status_code=401,
                detail="Missing or invalid Bearer authorization header",
                headers={"WWW-Authenticate": "Bearer"},
            )
        parts = authorization.split(maxsplit=1)
        if len(parts) != 2 or parts[0].lower() != "bearer":
            raise HTTPException(
                status_code=401,
                detail="Missing or invalid Bearer authorization header",
                headers={"WWW-Authenticate": "Bearer"},
            )
        token = parts[1].strip()
        if not secrets.compare_digest(token, api_token):
            raise HTTPException(
                status_code=401,
                detail="Invalid API token",
                headers={"WWW-Authenticate": "Bearer"},
            )

    @app.get("/health", response_model=HealthResponse)
    def health() -> HealthResponse:
        """Health check endpoint (no auth required)."""
        sam_loaded = bool(getattr(sam_engine, "loaded", False)) if sam_engine else False
        qwen_loaded = bool(getattr(assistant, "loaded", False)) if assistant else False
        asst_avail = bool(getattr(assistant, "available", False)) if assistant else False

        return HealthResponse(
            status="ok",
            gpu=gpu_name(),
            sam3_loaded=sam_loaded,
            qwen_loaded=qwen_loaded,
            prompt_assistant_available=asst_avail,
            busy=lock.locked(),
        )

    @app.post("/api/v1/assistant/prompt", response_model=PromptSuggestion)
    def assistant_prompt(
        image: UploadFile = File(...),
        user_request: str = Form(...),
        scope_preference: Optional[str] = Form("auto"),
        _: None = Depends(verify_auth),
    ) -> PromptSuggestion:
        """Ask prompt assistant to suggest refined prompts for SAM 3."""
        if assistant is None or not getattr(assistant, "available", False):
            raise HTTPException(status_code=503, detail="Prompt assistant is not available")

        # Validate user_request
        clean_request = user_request.strip() if user_request else ""
        if not clean_request:
            raise HTTPException(status_code=422, detail="user_request must not be empty")
        if len(clean_request) > 1000:
            raise HTTPException(status_code=422, detail="user_request exceeds maximum length of 1000 characters")

        # Validate scope_preference
        valid_scopes = {s.value for s in ScopePreference}
        scope_str = (scope_preference or "auto").strip()
        if scope_str not in valid_scopes:
            raise HTTPException(
                status_code=422,
                detail=f"Invalid scope_preference '{scope_str}'. Allowed: {sorted(valid_scopes)}",
            )
        scope_pref = ScopePreference(scope_str)

        # Validate uploaded image
        img = _read_and_validate_image(image, max_upload_mb)

        # Acquire lock non-blocking
        if not lock.acquire(blocking=False):
            raise HTTPException(status_code=409, detail="busy")

        try:
            return assistant.suggest(img, clean_request, scope_pref)
        except PromptAssistantUnavailable as exc:
            raise HTTPException(status_code=503, detail=str(exc))
        except PromptAssistantOutputError as exc:
            raise HTTPException(status_code=502, detail=str(exc))
        except HTTPException:
            raise
        except Exception:
            logger.exception("Unexpected error in prompt assistant inference")
            raise HTTPException(status_code=500, detail="Internal server error")
        finally:
            lock.release()

    @app.post("/api/v1/segment")
    def segment(
        image: UploadFile = File(...),
        mode: str = Form(...),
        prompt: Optional[str] = Form(None),
        bbox: Optional[str] = Form(None),
        confidence_threshold: Optional[float] = Form(None),
        export_profile: Optional[str] = Form("lean"),
        _: None = Depends(verify_auth),
    ) -> Response:
        """Run SAM 3 segmentation and return in-memory ZIP archive."""
        if sam_engine is None or not getattr(sam_engine, "loaded", False):
            raise HTTPException(status_code=503, detail="SAM 3 engine is not loaded")

        # Validate export_profile
        profile_clean = (export_profile or "lean").strip().lower()
        if profile_clean not in ("lean", "full"):
            raise HTTPException(
                status_code=422,
                detail=f"Invalid export_profile '{profile_clean}'. Allowed: ['full', 'lean']",
            )

        # Validate mode
        valid_modes = {m.value for m in Mode}
        mode_clean = (mode or "").strip()
        if mode_clean not in valid_modes:
            raise HTTPException(
                status_code=422,
                detail=f"Invalid mode '{mode_clean}'. Allowed: {sorted(valid_modes)}",
            )
        mode_enum = Mode(mode_clean)

        # Validate prompt
        clean_prompt = prompt.strip() if prompt else ""
        if mode_enum in (Mode.TEXT, Mode.TEXT_BOX) and not clean_prompt:
            raise HTTPException(status_code=422, detail="prompt is required for text and text_box modes")
        if mode_enum == Mode.BOX:
            clean_prompt = ""

        # Validate bbox
        parsed_bbox: Optional[list[float]] = None
        if mode_enum in (Mode.BOX, Mode.TEXT_BOX):
            if not bbox or not bbox.strip():
                raise HTTPException(status_code=422, detail="bbox is required for box and text_box modes")
            try:
                raw_bbox = json.loads(bbox)
            except Exception:
                raise HTTPException(status_code=422, detail="bbox must be a valid JSON array")

            if not isinstance(raw_bbox, (list, tuple)) or len(raw_bbox) != 4:
                raise HTTPException(status_code=422, detail="bbox must be an array of 4 numbers [cx, cy, w, h]")

            try:
                cx, cy, bw, bh = [float(v) for v in raw_bbox]
            except (TypeError, ValueError):
                raise HTTPException(status_code=422, detail="bbox values must be numeric")

            if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0):
                raise HTTPException(status_code=422, detail="bbox center coordinates cx, cy must be within [0, 1]")
            if not (0.0 < bw <= 1.0 and 0.0 < bh <= 1.0):
                raise HTTPException(status_code=422, detail="bbox dimensions w, h must be in (0, 1]")

            parsed_bbox = [cx, cy, bw, bh]

        # Validate confidence_threshold
        if confidence_threshold is not None:
            if not (0.0 < confidence_threshold <= 1.0):
                raise HTTPException(status_code=422, detail="confidence_threshold must be in (0, 1]")
            threshold_val = float(confidence_threshold)
        else:
            threshold_val = default_threshold

        # Validate image upload
        img = _read_and_validate_image(image, max_upload_mb)

        # Prepare export image (preserve RGBA if transparent PNG) and RGB image for SAM engine
        if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
            export_image = img.convert("RGBA")
        else:
            export_image = img.convert("RGB")
        engine_image = export_image.convert("RGB")

        # Acquire lock non-blocking
        if not lock.acquire(blocking=False):
            raise HTTPException(status_code=409, detail="busy")

        t0 = time.perf_counter()
        try:
            seg_result = sam_engine.segment(
                engine_image,
                mode=mode_enum,
                prompt=clean_prompt,
                bbox_cxcywh_norm=parsed_bbox,
                threshold=threshold_val,
            )
            infer_ms = max(1, int((time.perf_counter() - t0) * 1000))
            num_candidates_total = (
                len(seg_result.scores)
                if hasattr(seg_result, "scores") and hasattr(seg_result.scores, "__len__")
                else (
                    seg_result.masks.shape[0]
                    if hasattr(seg_result, "masks") and hasattr(seg_result.masks, "shape")
                    else 0
                )
            )

            t_export_start = time.perf_counter()
            zip_bytes = build_result_zip(
                export_image,
                seg_result,
                prompt=clean_prompt,
                mode=mode_enum,
                bbox_input=parsed_bbox,
                threshold=threshold_val,
                elapsed_ms=infer_ms,
                max_candidates=max_candidates,
                export_profile=profile_clean,
                overlay_max_side=overlay_max_side,
            )
            export_ms = max(0, int((time.perf_counter() - t_export_start) * 1000))
            zip_bytes_len = len(zip_bytes)

            # Safe logging: mode, prompt, profile, infer_ms, export_ms, zip_bytes, num_candidates_total (never images/tokens)
            logger.info(
                "segment finished: mode=%s prompt=%r profile=%s infer_ms=%d export_ms=%d zip_bytes=%d num_candidates_total=%d",
                mode_enum.value,
                clean_prompt,
                profile_clean,
                infer_ms,
                export_ms,
                zip_bytes_len,
                num_candidates_total,
            )

            return Response(
                content=zip_bytes,
                media_type="application/zip",
                headers={"Content-Disposition": 'attachment; filename="result.zip"'},
            )
        except ValueError as exc:
            raise HTTPException(status_code=422, detail=str(exc))
        except HTTPException:
            raise
        except Exception:
            logger.exception("Unexpected error during segmentation inference")
            raise HTTPException(status_code=500, detail="Internal server error")
        finally:
            lock.release()

    return app


In [ ]:
%%writefile /content/server_reference/errors.py
"""Error classes for the AI Artwork Extractor reference server.

Shared across api.py, fake_engines.py, and qwen_prompt_assistant.py.
"""
from __future__ import annotations


class PromptAssistantUnavailable(RuntimeError):
    """Raised when the prompt assistant model is unavailable or failed to load."""


class PromptAssistantOutputError(RuntimeError):
    """Raised when the prompt assistant model returns invalid or unparseable output."""


In [ ]:
%%writefile /content/server_reference/fake_engines.py
"""Deterministic fake engines for testing and local development without GPU.

Per contract §3 and project plan.
No torch dependency.
"""
from __future__ import annotations

from typing import TYPE_CHECKING, Any, List, Optional

import numpy as np

from server_reference.errors import (
    PromptAssistantOutputError,
    PromptAssistantUnavailable,
)
from server_reference.sam3_engine import SegResult
from server_reference.schemas import Mode, PromptSuggestion, Scope, ScopePreference

if TYPE_CHECKING:
    import PIL.Image


class FakeSam3Engine:
    """Deterministic fake SAM 3 engine for testing."""

    def __init__(self, device: str = "cpu", confidence_threshold: float = 0.35) -> None:
        self.device = device
        self.confidence_threshold = confidence_threshold
        self._loaded: bool = True

    def load(self) -> None:
        """Mark as loaded."""
        self._loaded = True

    @property
    def loaded(self) -> bool:
        """Return True if fake engine is loaded."""
        return self._loaded

    @loaded.setter
    def loaded(self, value: bool) -> None:
        self._loaded = bool(value)

    def segment(
        self,
        image: PIL.Image.Image,
        mode: Mode,
        prompt: Optional[str] = None,
        bbox_cxcywh_norm: Optional[List[float]] = None,
        threshold: Optional[float] = None,
    ) -> SegResult:
        """Return deterministic segmentation candidates without torch."""
        if not self._loaded:
            raise RuntimeError("FakeSam3Engine is not loaded.")

        w, h = image.width, image.height

        # Zero candidates case
        if prompt == "nothing":
            return SegResult(
                masks=np.zeros((0, h, w), dtype=bool),
                boxes_xyxy=np.zeros((0, 4), dtype=np.float32),
                scores=np.zeros((0,), dtype=np.float32),
            )

        if mode == Mode.TEXT:
            # 2 candidates: centered rectangle and centered ellipse
            masks = np.zeros((2, h, w), dtype=bool)

            # Candidate 0: Centered rectangle
            r_x1 = max(0, int(w * 0.25))
            r_x2 = min(w, int(w * 0.75))
            r_y1 = max(0, int(h * 0.25))
            r_y2 = min(h, int(h * 0.75))
            masks[0, r_y1:r_y2, r_x1:r_x2] = True

            # Candidate 1: Centered ellipse
            cx, cy = w / 2.0, h / 2.0
            rx, ry = max(1.0, w * 0.2), max(1.0, h * 0.2)
            y_coords, x_coords = np.ogrid[:h, :w]
            ellipse_mask = (((x_coords - cx) / rx) ** 2 + ((y_coords - cy) / ry) ** 2) <= 1.0
            masks[1] = ellipse_mask

            boxes_xyxy = np.array(
                [
                    [float(r_x1), float(r_y1), float(r_x2), float(r_y2)],
                    [float(cx - rx), float(cy - ry), float(cx + rx), float(cy + ry)],
                ],
                dtype=np.float32,
            )
            scores = np.array([0.92, 0.81], dtype=np.float32)

            return SegResult(
                masks=masks,
                boxes_xyxy=boxes_xyxy,
                scores=scores,
            )

        if mode in (Mode.BOX, Mode.TEXT_BOX):
            if bbox_cxcywh_norm is None:
                raise ValueError("bbox_cxcywh_norm is required for box and text_box modes")

            cx_n, cy_n, bw_n, bh_n = bbox_cxcywh_norm
            x1 = max(0, int((cx_n - bw_n / 2.0) * w))
            y1 = max(0, int((cy_n - bh_n / 2.0) * h))
            x2 = min(w, max(x1 + 1, int((cx_n + bw_n / 2.0) * w)))
            y2 = min(h, max(y1 + 1, int((cy_n + bh_n / 2.0) * h)))

            masks = np.zeros((1, h, w), dtype=bool)
            masks[0, y1:y2, x1:x2] = True

            boxes_xyxy = np.array([[float(x1), float(y1), float(x2), float(y2)]], dtype=np.float32)
            scores = np.array([0.95], dtype=np.float32)

            return SegResult(
                masks=masks,
                boxes_xyxy=boxes_xyxy,
                scores=scores,
            )

        raise ValueError(f"Unknown mode: {mode}")


class FakePromptAssistant:
    """Deterministic fake prompt assistant for testing."""

    def __init__(
        self,
        available: bool = True,
        loaded: bool = True,
        raise_unavailable: bool = False,
        raise_output_error: bool = False,
    ) -> None:
        self.available: bool = available
        self.loaded: bool = loaded
        self.raise_unavailable: bool = raise_unavailable
        self.raise_output_error: bool = raise_output_error

    def load(self) -> None:
        """Simulate loading."""
        if self.raise_unavailable:
            raise PromptAssistantUnavailable("Simulated load failure")
        self.loaded = True

    def unload(self) -> None:
        """Simulate unloading."""
        self.loaded = False

    def suggest(
        self,
        image: Any,
        user_request: str,
        scope_preference: ScopePreference = ScopePreference.AUTO,
    ) -> PromptSuggestion:
        """Generate deterministic prompt suggestions."""
        if not self.available or self.raise_unavailable:
            raise PromptAssistantUnavailable("Prompt assistant is currently unavailable")

        if self.raise_output_error:
            raise PromptAssistantOutputError("Assistant produced invalid output JSON")

        if scope_preference == ScopePreference.AUTO or str(scope_preference) == "auto":
            scope = Scope.ENTIRE_PATCH
        else:
            try:
                scope = Scope(scope_preference)
            except ValueError:
                scope = Scope.ENTIRE_PATCH

        return PromptSuggestion(
            target_description="Embroidered fabric patch on baseball cap",
            scope=scope,
            sam_prompt="embroidered fabric patch",
            alternative_prompts=["fabric patch", "embroidered logo patch"],
            location_hint="front center panel",
            needs_clarification=False,
            clarification_question="",
            reason="Extracted candidate patch based on user request",
        )


In [ ]:
%%writefile /content/server_reference/image_export.py
"""Image export and ZIP packaging for AI Artwork Extractor.

Produces transparent cutouts, overlays, masks, and metadata.json
strictly following contract §2 and project plan §5.4.
"""
from __future__ import annotations

import io
import zipfile
from typing import TYPE_CHECKING, Any, List, Optional

import numpy as np
from PIL import Image

from server_reference.schemas import CandidateMeta, Mode, SegmentMetadata

if TYPE_CHECKING:
    import PIL.Image
    from server_reference.sam3_engine import SegResult


def normalize_masks(masks_input: Any, target_size: tuple[int, int]) -> np.ndarray:
    """Normalize masks array/tensor of shape [N,1,H,W], [N,H,W], or [H,W] to bool [N, target_H, target_W].

    Resizes to target_size (width, height) using NEAREST interpolation if dimensions differ.
    """
    if hasattr(masks_input, "detach"):
        arr = masks_input.detach().float().cpu().numpy()
    else:
        arr = np.asarray(masks_input)

    # Convert to float/bool
    if arr.dtype == bool:
        pass
    elif np.issubdtype(arr.dtype, np.floating):
        arr = arr > 0.5
    else:
        arr = arr != 0

    if arr.ndim == 4 and arr.shape[1] == 1:
        arr = arr[:, 0]
    elif arr.ndim == 2:
        arr = arr[None, ...]
    elif arr.ndim == 3:
        pass
    else:
        raise ValueError(f"Unsupported mask shape: {arr.shape}")

    arr = arr.astype(bool)
    target_w, target_h = target_size
    n_instances = arr.shape[0]

    if n_instances == 0:
        return np.zeros((0, target_h, target_w), dtype=bool)

    cur_h, cur_w = arr.shape[1], arr.shape[2]
    if (cur_h, cur_w) == (target_h, target_w):
        return arr

    resized = np.zeros((n_instances, target_h, target_w), dtype=bool)
    for i in range(n_instances):
        pil_mask = Image.fromarray(arr[i].astype(np.uint8) * 255, mode="L")
        resized_pil = pil_mask.resize((target_w, target_h), resample=Image.Resampling.NEAREST)
        resized[i] = np.array(resized_pil) > 127

    return resized


def build_result_zip(
    image: PIL.Image.Image,
    result: SegResult,
    *,
    prompt: str,
    mode: Mode,
    bbox_input: Optional[List[float]],
    threshold: float,
    elapsed_ms: int,
    crop_padding_px: int = 12,
    max_candidates: int = 12,
    export_profile: str = "lean",
    overlay_max_side: int = 768,
) -> bytes:
    """Export segmentation results into an in-memory ZIP package per contract §2.

    Returns the raw ZIP bytes without creating any temporary files on disk.
    Supports export profiles:
      - 'lean' (default): mask, cutout_crop, downscaled overlay (max side <= overlay_max_side),
        and NO full-canvas cutout.
      - 'full': original behavior with full-canvas cutout and full-res overlay.
    """
    profile = (export_profile or "lean").strip().lower()
    if profile not in ("lean", "full"):
        raise ValueError(f"Invalid export_profile '{export_profile}'. Allowed: ['full', 'lean']")

    w, h = image.width, image.height
    masks = normalize_masks(result.masks, (w, h))
    num_candidates_total = masks.shape[0]

    # Scores and boxes
    scores = (
        result.scores.reshape(-1)
        if hasattr(result.scores, "reshape")
        else np.array(result.scores, dtype=np.float32)
    )
    boxes = (
        result.boxes_xyxy
        if hasattr(result.boxes_xyxy, "ndim") and result.boxes_xyxy.ndim == 2
        else np.zeros((num_candidates_total, 4), dtype=np.float32)
    )

    # Sort candidate indices by score descending
    if num_candidates_total > 0 and len(scores) >= num_candidates_total:
        order = np.argsort(-scores[:num_candidates_total])
    else:
        order = np.arange(num_candidates_total)

    if max_candidates is not None and max_candidates > 0:
        order = order[:max_candidates]

    # Prepare base RGBA numpy array
    if image.mode == "RGBA":
        source_rgba = np.array(image)
    else:
        source_rgba = np.array(image.convert("RGBA"))

    orig_alpha = source_rgba[:, :, 3].astype(np.uint16)
    has_orig_transparency = bool((orig_alpha < 255).any())
    base_rgb = source_rgba[:, :, :3]
    tint = np.array([255, 40, 170], dtype=np.uint16)

    # In lean profile, pre-downscale base RGB image for overlay review previews
    max_dim = max(w, h)
    if profile == "lean" and overlay_max_side and max_dim > overlay_max_side:
        scale = overlay_max_side / float(max_dim)
        dw = max(1, int(round(w * scale)))
        dh = max(1, int(round(h * scale)))
        base_pil = Image.fromarray(base_rgb, mode="RGB")
        downscaled_base_rgb = np.array(
            base_pil.resize((dw, dh), resample=Image.Resampling.BILINEAR)
        )
    else:
        dw, dh = w, h
        downscaled_base_rgb = None

    candidates_meta: List[CandidateMeta] = []
    zip_buffer = io.BytesIO()

    with zipfile.ZipFile(zip_buffer, mode="w", compression=zipfile.ZIP_STORED) as zf:
        for new_id, orig_idx in enumerate(order):
            mask = masks[orig_idx]
            mask_u8 = mask.astype(np.uint8) * 255
            score_val = float(scores[orig_idx]) if orig_idx < len(scores) else 0.0

            # 1. Mask PNG (L mode, 0/255)
            mask_filename = f"candidate_{new_id:02d}_mask.png"
            mask_pil = Image.fromarray(mask_u8, mode="L")
            mask_bytes = io.BytesIO()
            mask_pil.save(mask_bytes, format="PNG", compress_level=1)
            zf.writestr(f"result/{mask_filename}", mask_bytes.getvalue())

            # 2. Cutout Crop (clamped to image; absent if mask is empty)
            ys, xs = np.where(mask)
            has_crop = len(xs) > 0
            crop_filename: Optional[str] = None
            crop_box_xyxy_px: Optional[List[int]] = None
            if has_crop:
                x0 = max(0, int(xs.min()) - crop_padding_px)
                x1 = min(w, int(xs.max()) + crop_padding_px + 1)
                y0 = max(0, int(ys.min()) - crop_padding_px)
                y1 = min(h, int(ys.max()) + crop_padding_px + 1)
                crop_box_xyxy_px = [x0, y0, x1, y1]

                crop_rgba = source_rgba[y0:y1, x0:x1].copy()
                mask_crop = mask_u8[y0:y1, x0:x1]
                if has_orig_transparency:
                    crop_rgba[:, :, 3] = (
                        (crop_rgba[:, :, 3].astype(np.uint16) * mask_crop.astype(np.uint16)) // 255
                    ).astype(np.uint8)
                else:
                    crop_rgba[:, :, 3] = mask_crop

                crop_pil = Image.fromarray(crop_rgba, mode="RGBA")
                crop_filename = f"candidate_{new_id:02d}_cutout_crop.png"
                crop_bytes = io.BytesIO()
                crop_pil.save(crop_bytes, format="PNG", compress_level=1)
                zf.writestr(f"result/{crop_filename}", crop_bytes.getvalue())

            # 3. Overlay PNG (RGB colored mask review preview)
            overlay_filename = f"candidate_{new_id:02d}_overlay.png"
            overlay_bytes = io.BytesIO()
            if downscaled_base_rgb is not None:
                # Lean profile: compute on downscaled image
                d_mask = np.array(
                    mask_pil.resize((dw, dh), resample=Image.Resampling.NEAREST)
                ) > 127
                overlay_arr = downscaled_base_rgb.copy()
                overlay_arr[d_mask] = (
                    (downscaled_base_rgb[d_mask].astype(np.uint16) * 145 + tint * 110 + 127) // 255
                ).astype(np.uint8)
                overlay_pil = Image.fromarray(overlay_arr, mode="RGB")
            else:
                # Full profile or small image: full resolution overlay
                overlay_arr = base_rgb.copy()
                overlay_arr[mask] = (
                    (base_rgb[mask].astype(np.uint16) * 145 + tint * 110 + 127) // 255
                ).astype(np.uint8)
                overlay_pil = Image.fromarray(overlay_arr, mode="RGB")

            overlay_pil.save(overlay_bytes, format="PNG", compress_level=1)
            zf.writestr(f"result/{overlay_filename}", overlay_bytes.getvalue())

            # 4. Full Canvas Cutout (full profile ONLY)
            cutout_filename: Optional[str] = None
            if profile == "full":
                cutout_arr = source_rgba.copy()
                if has_orig_transparency:
                    cutout_arr[:, :, 3] = (
                        (orig_alpha * mask_u8.astype(np.uint16)) // 255
                    ).astype(np.uint8)
                else:
                    cutout_arr[:, :, 3] = mask_u8

                cutout_filename = f"candidate_{new_id:02d}_cutout.png"
                cutout_pil = Image.fromarray(cutout_arr, mode="RGBA")
                cutout_bytes = io.BytesIO()
                cutout_pil.save(cutout_bytes, format="PNG", compress_level=1)
                zf.writestr(f"result/{cutout_filename}", cutout_bytes.getvalue())

            # Candidate bbox
            if orig_idx < len(boxes) and len(boxes[orig_idx]) == 4:
                bbox_xyxy = [float(b) for b in boxes[orig_idx]]
            elif has_crop:
                bbox_xyxy = [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]
            else:
                bbox_xyxy = [0.0, 0.0, 0.0, 0.0]

            files_dict = {
                "mask": mask_filename,
                "overlay": overlay_filename,
            }
            if cutout_filename is not None:
                files_dict["cutout"] = cutout_filename
            if crop_filename is not None:
                files_dict["cutout_crop"] = crop_filename

            candidate_meta = CandidateMeta(
                id=new_id,
                score=score_val,
                bbox_xyxy_px=bbox_xyxy,
                mask_area_px=int(np.sum(mask)),
                crop_box_xyxy_px=crop_box_xyxy_px,
                files=files_dict,
            )
            candidates_meta.append(candidate_meta)

        # 5. SegmentMetadata
        metadata = SegmentMetadata(
            prompt=prompt,
            mode=mode,
            bbox_input=bbox_input,
            image_size=[w, h],
            confidence_threshold=threshold,
            crop_padding_px=crop_padding_px,
            export_profile=profile,
            overlay_max_side=overlay_max_side if profile == "lean" else None,
            elapsed_ms=elapsed_ms,
            num_candidates=len(candidates_meta),
            num_candidates_total=num_candidates_total,
            candidates=candidates_meta,
        )

        metadata_json = metadata.model_dump_json(indent=2)
        zf.writestr("result/metadata.json", metadata_json.encode("utf-8"))

    return zip_buffer.getvalue()


In [ ]:
%%writefile /content/server_reference/qwen_prompt_assistant.py
"""Qwen3-VL-4B Prompt Assistant for Meta SAM 3 Segmentation.

Converts designer natural-language requests into concise object concepts
and noun phrases for SAM 3 text prompts.
"""
from __future__ import annotations

import io
import json
import os
import re
from typing import Any, List, Optional

from server_reference.schemas import PromptSuggestion, Scope, ScopePreference

try:
    from server_reference.errors import (
        PromptAssistantUnavailable,
        PromptAssistantOutputError,
    )
except ImportError:  # pragma: no cover - until server worker merges
    class PromptAssistantUnavailable(RuntimeError):
        pass

    class PromptAssistantOutputError(RuntimeError):
        pass

__all__ = [
    "PromptAssistantUnavailable",
    "PromptAssistantOutputError",
    "SYSTEM_PROMPT",
    "build_user_message",
    "parse_suggestion",
    "QwenPromptAssistant",
]

SYSTEM_PROMPT = """You are an assistant that converts a designer's natural-language
request into concise object concepts for Meta SAM 3 segmentation.

You receive:
1. A product image.
2. A request written by the designer, usually in Vietnamese.

Your job is NOT to segment the image. Your job is to identify the
intended visible object and propose short text prompts for SAM 3.

Rules:
- Preserve the designer's intended extraction scope.
- Distinguish entire_patch, inner_artwork, and individual_decoration.
- If the user asks for the whole patch, describe the patch itself,
  not only the artwork inside it.
- If the user asks for the inner artwork, describe only that inner
  artwork.
- sam_prompt and alternative_prompts must ALWAYS be short English noun phrases
  even when the request is in Vietnamese (e.g. "lấy bông hoa đỏ trên túi" ->
  embroidered red poppy flowers), because SAM 3 only understands English.
- target_description, reason, and clarification_question may be in Vietnamese.
- Prefer short noun phrases, not long natural-language instructions.
- Describe one concept per prompt.
- Do not combine multiple object categories into one prompt.
- Do not invent hidden, occluded, or unseen details.
- Do not rewrite or recreate lettering, logos, or artwork.
- Do not generate bounding-box coordinates.
- Provide up to 3 alternative prompts only when useful.
- If the target is ambiguous, explain briefly and set needs_clarification=true.
- The user must approve or edit the suggested prompt before segmentation.

Return valid JSON only, using this schema:
{
  "target_description": "short description",
  "scope": "entire_patch | inner_artwork | individual_decoration | unknown",
  "sam_prompt": "short noun phrase for SAM 3",
  "alternative_prompts": ["prompt 1", "prompt 2"],
  "location_hint": "short visible location or empty string",
  "needs_clarification": false,
  "clarification_question": "empty string or one short question",
  "reason": "brief explanation"
}"""


def build_user_message(
    user_request: str,
    scope_preference: ScopePreference | str = ScopePreference.AUTO,
) -> str:
    """Format user message for the assistant prompt.

    Contains the designer request; if scope_preference != auto,
    adds a line saying the designer prefers that scope.
    """
    user_req = (user_request or "").strip()
    if isinstance(scope_preference, ScopePreference):
        pref_val = scope_preference.value
    elif scope_preference is not None:
        pref_val = str(scope_preference).strip().lower()
    else:
        pref_val = ScopePreference.AUTO.value

    lines: list[str] = []
    if user_req:
        lines.append(user_req)
    if pref_val and pref_val != ScopePreference.AUTO.value:
        lines.append(f"The designer prefers that scope: {pref_val}.")
    return "\n".join(lines)


def _extract_first_json_object(text: str) -> str:
    """Find the first balanced {...} substring using string-aware brace matching."""
    start_idx = text.find("{")
    if start_idx == -1:
        raise PromptAssistantOutputError("No JSON object found in output text")

    depth = 0
    in_string = False
    escape = False

    for i in range(start_idx, len(text)):
        char = text[i]
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    return text[start_idx : i + 1]

    raise PromptAssistantOutputError("Unbalanced braces in JSON object")


def parse_suggestion(text: str) -> PromptSuggestion:
    """Parse raw LLM output into a validated PromptSuggestion instance.

    Strips ```json fences, finds the first balanced {...} object,
    validates with PromptSuggestion schema, converts invalid scopes to unknown,
    and checks the empty sam_prompt rule.
    """
    if not isinstance(text, str) or not text.strip():
        raise PromptAssistantOutputError("Assistant output is empty or not text")

    # Strip ```json and ``` fences
    cleaned = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE)

    # Find the first balanced {...} object (string-aware brace matching)
    json_str = _extract_first_json_object(cleaned)

    try:
        data = json.loads(json_str)
    except Exception as e:
        raise PromptAssistantOutputError(f"Failed to decode JSON: {e}") from e

    if not isinstance(data, dict):
        raise PromptAssistantOutputError(f"Expected a JSON object, got {type(data).__name__}")

    # Unknown/invalid scope -> unknown
    valid_scopes = {s.value for s in Scope}
    raw_scope = data.get("scope")
    if isinstance(raw_scope, Scope):
        pass
    elif isinstance(raw_scope, str) and raw_scope.strip().lower() in valid_scopes:
        data["scope"] = Scope(raw_scope.strip().lower())
    else:
        data["scope"] = Scope.UNKNOWN

    # Filter non-ASCII alternative prompts before schema validation
    if isinstance(data.get("alternative_prompts"), list):
        data["alternative_prompts"] = [
            p for p in data["alternative_prompts"]
            if isinstance(p, str) and p.isascii()
        ]

    try:
        suggestion = PromptSuggestion.model_validate(data)
    except Exception as e:
        raise PromptAssistantOutputError(f"Schema validation error: {e}") from e

    # Non-ASCII alternative prompts are dropped (if any remain)
    suggestion.alternative_prompts = [
        p for p in suggestion.alternative_prompts if p.isascii()
    ]

    # Empty or non-ASCII sam_prompt while needs_clarification is false -> invalid
    if not suggestion.needs_clarification:
        if not suggestion.sam_prompt:
            raise PromptAssistantOutputError(
                "sam_prompt cannot be empty when needs_clarification is False"
            )
        if not suggestion.sam_prompt.isascii():
            raise PromptAssistantOutputError(
                "sam_prompt and alternative_prompts must be in English"
            )

    return suggestion


class QwenPromptAssistant:
    """Assistant powered by Qwen3-VL-4B-Instruct for generating SAM 3 prompts."""

    def __init__(
        self,
        model_id: str = "Qwen/Qwen3-VL-4B-Instruct",
        enabled: bool = True,
        min_free_vram_gb: float = 5.0,
        max_image_side: int = 768,
        max_new_tokens: int = 160,
        device: str = "cuda",
    ) -> None:
        self.model_id = model_id
        self.enabled = bool(enabled)
        self.min_free_vram_gb = float(min_free_vram_gb)
        self.max_image_side = int(max_image_side)
        self.max_new_tokens = int(max_new_tokens)
        self.device = device

        self._available = bool(enabled)
        self.last_error: str = ""
        self.model: Any = None
        self.processor: Any = None

    @property
    def available(self) -> bool:
        """Return True if enabled and not disabled by an error or OOM."""
        return self._available and self.enabled

    @available.setter
    def available(self, val: bool) -> None:
        self._available = bool(val)

    @property
    def loaded(self) -> bool:
        """Return True if model and processor are loaded."""
        return self.model is not None and self.processor is not None

    def load(self) -> None:
        """Load model and processor into GPU memory with 4-bit NF4 quantization."""
        if not self.available:
            raise PromptAssistantUnavailable(
                f"Prompt assistant is unavailable (last error: {self.last_error or 'disabled'})."
            )
        if self.loaded:
            return

        try:
            import torch
            from transformers import AutoProcessor, BitsAndBytesConfig

            try:
                from transformers import Qwen3VLForConditionalGeneration
            except ImportError:
                from transformers import (  # type: ignore[assignment]
                    AutoModelForImageTextToText as Qwen3VLForConditionalGeneration,
                )

            if not torch.cuda.is_available():
                raise RuntimeError("CUDA is not available")

            free_bytes, _ = torch.cuda.mem_get_info()
            free_gb = free_bytes / (1024 ** 3)
            if free_gb < self.min_free_vram_gb:
                raise RuntimeError(
                    f"Insufficient VRAM: {free_gb:.2f} GB free, required {self.min_free_vram_gb:.2f} GB"
                )

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )

            self.processor = AutoProcessor.from_pretrained(self.model_id)
            self.model = Qwen3VLForConditionalGeneration.from_pretrained(
                self.model_id,
                quantization_config=bnb_config,
                device_map="cuda:0",
                dtype=torch.float16,
            )
            self.model.eval()
        except Exception as e:
            self.unload()
            self.available = False
            self.last_error = f"Failed to load Qwen model '{self.model_id}': {e}"
            raise PromptAssistantUnavailable(self.last_error) from e

    def unload(self) -> None:
        """Unload model and processor from memory and release cached VRAM."""
        self.model = None
        self.processor = None
        import gc

        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    def _is_oom_error(self, e: BaseException) -> bool:
        """Check if an exception represents a CUDA Out Of Memory condition."""
        try:
            import torch

            if isinstance(e, torch.cuda.OutOfMemoryError):
                return True
        except (ImportError, AttributeError):
            pass

        msg = str(e).lower()
        return "out of memory" in msg or "cuda oom" in msg or "outofmemoryerror" in type(e).__name__.lower()

    @staticmethod
    def _prepare_image(image: Any, max_side: int) -> Any:
        """Ensure image is an RGB PIL Image with max side <= max_side keeping aspect ratio."""
        from PIL import Image

        if not isinstance(image, Image.Image):
            if isinstance(image, (bytes, bytearray)):
                image = Image.open(io.BytesIO(image))
            elif isinstance(image, (str, os.PathLike)):
                image = Image.open(image)
            else:
                raise ValueError(f"Unsupported image type: {type(image)}")

        if image.mode != "RGB":
            image = image.convert("RGB")

        w, h = image.size
        max_dim = max(w, h)
        if max_dim > max_side:
            scale = max_side / max_dim
            new_w = max(1, int(round(w * scale)))
            new_h = max(1, int(round(h * scale)))
            image = image.resize((new_w, new_h), Image.Resampling.LANCZOS)

        return image

    def _generate(self, messages: list[dict[str, Any]]) -> str:
        """Execute model generation using Hugging Face transformers.

        Overridable in subclasses for testing without GPU/torch.
        """
        import torch

        if not self.loaded:
            self.load()

        with torch.inference_mode():
            inputs = self.processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            )
            device = getattr(self.model, "device", self.device)
            model_dtype = getattr(self.model, "dtype", torch.float16)
            inputs = {
                k: (
                    v.to(device, dtype=model_dtype)
                    if hasattr(v, "is_floating_point") and v.is_floating_point()
                    else (v.to(device) if hasattr(v, "to") else v)
                )
                for k, v in inputs.items()
            }
            prompt_len = inputs["input_ids"].shape[1]
            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
            trimmed_ids = generated_ids[:, prompt_len:]
            output_texts = self.processor.batch_decode(
                trimmed_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
            del inputs, generated_ids
            return output_texts[0] if output_texts else ""

    def suggest(
        self,
        image: Any,
        user_request: str,
        scope_preference: ScopePreference | str = ScopePreference.AUTO,
    ) -> PromptSuggestion:
        """Suggest SAM 3 prompts from an image and designer request.

        Lazy-loads model if needed. Retries once if the model produces
        invalid JSON. Disables assistant on OOM.
        """
        if not self.available:
            raise PromptAssistantUnavailable(
                f"Prompt assistant is unavailable (last error: {self.last_error or 'disabled'})."
            )

        if not self.loaded:
            self.load()

        prepared_image = self._prepare_image(image, self.max_image_side)
        user_text = build_user_message(user_request, scope_preference)

        messages: list[dict[str, Any]] = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": prepared_image},
                    {"type": "text", "text": user_text},
                ],
            },
        ]

        # First attempt
        first_err: Optional[Exception] = None
        try:
            raw_text = self._generate(messages)
        except Exception as e:
            if self._is_oom_error(e):
                self.unload()
                self.available = False
                self.last_error = f"CUDA Out of Memory during generation: {e}"
                raise PromptAssistantUnavailable(self.last_error) from e
            raise

        try:
            return parse_suggestion(raw_text)
        except PromptAssistantOutputError as err:
            first_err = err

        # Retry once: assistant turn with previous answer, user turn with schema reminder
        retry_instruction = "Return valid JSON only, matching the schema."
        if first_err and "English" in str(first_err):
            retry_instruction = (
                "sam_prompt and alternative_prompts must be in English. "
                "Return valid JSON only, matching the schema."
            )

        retry_messages = list(messages) + [
            {
                "role": "assistant",
                "content": [{"type": "text", "text": raw_text}],
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": retry_instruction}],
            },
        ]

        try:
            retry_raw_text = self._generate(retry_messages)
        except Exception as e:
            if self._is_oom_error(e):
                self.unload()
                self.available = False
                self.last_error = f"CUDA Out of Memory during generation retry: {e}"
                raise PromptAssistantUnavailable(self.last_error) from e
            raise

        try:
            return parse_suggestion(retry_raw_text)
        except PromptAssistantOutputError as err:
            raise PromptAssistantOutputError(
                f"Prompt assistant failed to produce valid JSON after retry: {err}"
            ) from err

    def vram_report(self) -> dict[str, float]:
        """Report GPU VRAM usage in gigabytes, or empty dict if CUDA is unavailable."""
        try:
            import torch

            if not torch.cuda.is_available():
                return {}
            free_bytes, total_bytes = torch.cuda.mem_get_info()
            allocated_bytes = torch.cuda.memory_allocated()
            gb = 1024 ** 3
            alloc_gb = round(allocated_bytes / gb, 3)
            free_gb = round(free_bytes / gb, 3)
            total_gb = round(total_bytes / gb, 3)
            return {
                "allocated_gb": alloc_gb,
                "free_gb": free_gb,
                "total_gb": total_gb,
                "allocated": alloc_gb,
                "free": free_gb,
                "total": total_gb,
            }
        except Exception:
            return {}


In [ ]:
%%writefile /content/server_reference/run_local_fake.py
"""Run local reference server with deterministic fake engines.

Enables local development of UI and integration tests without GPU.
Usage:
  python -m server_reference.run_local_fake [--port 8000] [--token TOKEN]
"""
from __future__ import annotations

import argparse
import os
import secrets
import sys

import uvicorn

from server_reference.api import create_app
from server_reference.fake_engines import FakePromptAssistant, FakeSam3Engine


def main() -> None:
    """Parse arguments and start Uvicorn with fake engines."""
    parser = argparse.ArgumentParser(description="Run AI Artwork Extractor fake server locally")
    parser.add_argument("--port", type=int, default=8000, help="Port to listen on (default 8000)")
    parser.add_argument("--host", type=str, default="127.0.0.1", help="Host to listen on (default 127.0.0.1)")
    parser.add_argument("--token", type=str, default=None, help="API bearer token (optional)")
    parser.add_argument(
        "--cors-origin",
        action="append",
        default=None,
        help="Allowed CORS origin (repeatable, defaults to http://localhost:5173 and http://127.0.0.1:5173)",
    )

    args = parser.parse_args()

    token = args.token or os.environ.get("API_TOKEN")
    if not token:
        token = secrets.token_urlsafe(32)
        print(f"\n[LOCAL DEV ONLY] Generated bearer token: {token}\n", flush=True)
    else:
        print("\n[LOCAL DEV ONLY] Using token from --token/API_TOKEN\n", flush=True)

    fake_sam = FakeSam3Engine(device="cpu")
    fake_assistant = FakePromptAssistant(available=True, loaded=True)

    cors_origins = (
        args.cors_origin
        if args.cors_origin is not None
        else ["http://localhost:5173", "http://127.0.0.1:5173"]
    )

    app = create_app(
        sam_engine=fake_sam,
        assistant=fake_assistant,
        api_token=token,
        cors_origins=cors_origins,
    )

    print(f"Starting server on http://{args.host}:{args.port}")
    print(f"Health check: http://{args.host}:{args.port}/health\n")
    uvicorn.run(app, host=args.host, port=args.port)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/server_reference/sam3_engine.py
"""SAM 3 model engine wrapper for AI Artwork Extractor.

Per contract §3 and project plan §5.
Provides SegResult dataclass and Sam3Engine class with lazy imports
so CPU-only test environments without torch/sam3 can import and run.
"""
from __future__ import annotations

import contextlib
from dataclasses import dataclass
from typing import TYPE_CHECKING, Any, List, Optional

import numpy as np
from PIL import Image

from server_reference.schemas import Mode

if TYPE_CHECKING:
    import PIL.Image


@dataclass
class SegResult:
    """Segmentation output dataclass per contract §3."""

    masks: np.ndarray  # bool [N, H, W] at ORIGINAL image size
    boxes_xyxy: np.ndarray  # float32 [N, 4] pixel coords
    scores: np.ndarray  # float32 [N]


def gpu_name() -> Optional[str]:
    """Return the GPU model name or None if CUDA/torch is unavailable."""
    try:
        import torch

        if torch.cuda.is_available():
            return str(torch.cuda.get_device_name(0))
    except Exception:
        pass
    return None


def _to_numpy_masks(mask_tensor: Any, target_size: tuple[int, int]) -> np.ndarray:
    """Convert tensor/array masks to boolean [N, H, W] numpy array matching target_size (width, height)."""
    if hasattr(mask_tensor, "detach"):
        arr = mask_tensor.detach().float().cpu().numpy()
    else:
        arr = np.asarray(mask_tensor, dtype=np.float32)

    if arr.ndim == 4 and arr.shape[1] == 1:
        arr = arr[:, 0]
    elif arr.ndim == 2:
        arr = arr[None, ...]
    elif arr.ndim == 3:
        pass
    else:
        raise ValueError(f"Unexpected mask tensor shape: {arr.shape}")

    bool_masks = arr > 0.5
    target_w, target_h = target_size
    n_instances = bool_masks.shape[0]

    if n_instances == 0:
        return np.zeros((0, target_h, target_w), dtype=bool)

    cur_h, cur_w = bool_masks.shape[1], bool_masks.shape[2]
    if (cur_h, cur_w) == (target_h, target_w):
        return bool_masks

    # Resize each mask to original size using NEAREST interpolation
    resized_masks = np.zeros((n_instances, target_h, target_w), dtype=bool)
    for i in range(n_instances):
        pil_mask = Image.fromarray(bool_masks[i].astype(np.uint8) * 255, mode="L")
        resized_pil = pil_mask.resize((target_w, target_h), resample=Image.Resampling.NEAREST)
        resized_masks[i] = np.array(resized_pil) > 127

    return resized_masks


class Sam3Engine:
    """SAM 3 inference engine wrapper."""

    def __init__(self, device: str = "cuda", confidence_threshold: float = 0.35) -> None:
        self.device = device
        self.confidence_threshold = confidence_threshold
        self._model: Any = None
        self._processor: Any = None

    def load(self) -> None:
        """Lazily load SAM 3 model and processor onto target device."""
        if self._processor is not None:
            return

        from sam3.model_builder import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor

        self._model = build_sam3_image_model(device=self.device, eval_mode=True)
        self._model.eval()
        self._processor = Sam3Processor(
            self._model,
            device=self.device,
            confidence_threshold=self.confidence_threshold,
        )

    @property
    def loaded(self) -> bool:
        """Return True if model and processor are initialized."""
        return self._processor is not None

    def segment(
        self,
        image: PIL.Image.Image,
        mode: Mode,
        prompt: Optional[str] = None,
        bbox_cxcywh_norm: Optional[List[float]] = None,
        threshold: Optional[float] = None,
    ) -> SegResult:
        """Run segmentation inference on image using given mode and prompts."""
        if not self.loaded:
            raise RuntimeError("Sam3Engine is not loaded. Call load() first.")

        import torch

        target_thresh = threshold if threshold is not None else self.confidence_threshold
        has_thresh_attr = hasattr(self._processor, "confidence_threshold")
        orig_thresh = getattr(self._processor, "confidence_threshold", None) if has_thresh_attr else None

        try:
            if has_thresh_attr and threshold is not None:
                self._processor.confidence_threshold = target_thresh

            if self.device.startswith("cuda") and torch.cuda.is_available():
                amp_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
            else:
                amp_ctx = contextlib.nullcontext()

            with torch.inference_mode(), amp_ctx:
                # Fresh state per call: never reuse old prompts
                image_state = self._processor.set_image(image)

                if mode == Mode.TEXT:
                    if not prompt or not prompt.strip():
                        raise ValueError("Text mode requires non-empty prompt")
                    output = self._processor.set_text_prompt(state=image_state, prompt=prompt.strip())
                elif mode == Mode.BOX:
                    if bbox_cxcywh_norm is None:
                        raise ValueError("Box mode requires bbox_cxcywh_norm")
                    output = self._processor.add_geometric_prompt(
                        state=image_state,
                        box=bbox_cxcywh_norm,
                        label=True,
                    )
                elif mode == Mode.TEXT_BOX:
                    if not prompt or not prompt.strip():
                        raise ValueError("Text+Box mode requires non-empty prompt")
                    if bbox_cxcywh_norm is None:
                        raise ValueError("Text+Box mode requires bbox_cxcywh_norm")
                    self._processor.set_text_prompt(state=image_state, prompt=prompt.strip())
                    output = self._processor.add_geometric_prompt(
                        state=image_state,
                        box=bbox_cxcywh_norm,
                        label=True,
                    )
                else:
                    raise ValueError(f"Unsupported mode: {mode}")

            w, h = image.width, image.height
            if output is None or "scores" not in output or len(output["scores"]) == 0:
                return SegResult(
                    masks=np.zeros((0, h, w), dtype=bool),
                    boxes_xyxy=np.zeros((0, 4), dtype=np.float32),
                    scores=np.zeros((0,), dtype=np.float32),
                )

            masks = _to_numpy_masks(output["masks"], (w, h))
            boxes = output["boxes"].detach().float().cpu().numpy().astype(np.float32)
            scores = output["scores"].detach().float().cpu().numpy().reshape(-1).astype(np.float32)

            return SegResult(
                masks=masks,
                boxes_xyxy=boxes,
                scores=scores,
            )
        finally:
            if has_thresh_attr and orig_thresh is not None:
                self._processor.confidence_threshold = orig_thresh


In [ ]:
%%writefile /content/server_reference/schemas.py
"""Shared Pydantic schemas for the AI Artwork Extractor API.

This file is the single source of truth shared by the Colab server
(server_reference/*) and the React web UI (web_app/src/core/types.ts mirrors these models).
Do NOT change it without coordinator approval — see docs/API_CONTRACT.md.
"""
from __future__ import annotations

from enum import Enum
from typing import List, Optional

from pydantic import BaseModel, Field, field_validator


class Mode(str, Enum):
    TEXT = "text"
    BOX = "box"
    TEXT_BOX = "text_box"


class Scope(str, Enum):
    ENTIRE_PATCH = "entire_patch"
    INNER_ARTWORK = "inner_artwork"
    INDIVIDUAL_DECORATION = "individual_decoration"
    UNKNOWN = "unknown"


class ScopePreference(str, Enum):
    AUTO = "auto"
    ENTIRE_PATCH = "entire_patch"
    INNER_ARTWORK = "inner_artwork"
    INDIVIDUAL_DECORATION = "individual_decoration"


class PromptSuggestion(BaseModel):
    """Qwen Prompt Assistant output (plan section 4.4)."""

    target_description: str = ""
    scope: Scope = Scope.UNKNOWN
    sam_prompt: str = ""
    alternative_prompts: List[str] = Field(default_factory=list, max_length=3)
    location_hint: str = ""
    needs_clarification: bool = False
    clarification_question: str = ""
    reason: str = ""

    @field_validator("alternative_prompts", mode="before")
    @classmethod
    def _clean_alternatives(cls, v):
        if v is None:
            return []
        if isinstance(v, str):
            v = [v]
        out = []
        for p in v:
            p = str(p).strip()
            if p and p not in out:
                out.append(p)
        return out[:3]

    @field_validator("sam_prompt", "target_description", "location_hint",
                     "clarification_question", "reason", mode="before")
    @classmethod
    def _strip(cls, v):
        return "" if v is None else str(v).strip()


class HealthResponse(BaseModel):
    status: str = "ok"
    gpu: Optional[str] = None
    sam3_loaded: bool = False
    qwen_loaded: bool = False
    prompt_assistant_available: bool = False
    busy: bool = False


class CandidateMeta(BaseModel):
    id: int
    score: float
    bbox_xyxy_px: List[float] = Field(..., min_length=4, max_length=4)
    mask_area_px: int
    crop_box_xyxy_px: Optional[List[int]] = Field(
        default=None,
        description="[x0, y0, x1, y1) of cutout_crop inside the original image (exclusive x1/y1); null if mask empty",
    )
    files: dict = Field(
        default_factory=dict,
        description="Relative names inside result/: mask, overlay, cutout, cutout_crop (may be absent if mask empty)",
    )


class SegmentMetadata(BaseModel):
    prompt: str = ""
    mode: Mode
    bbox_input: Optional[List[float]] = Field(
        default=None, description="Normalized [cx, cy, w, h] as received, or null for text mode"
    )
    image_size: List[int] = Field(..., min_length=2, max_length=2, description="[width, height]")
    confidence_threshold: float
    crop_padding_px: int = 12
    export_profile: str = Field(default="lean", description="'lean' (default) or 'full'")
    overlay_max_side: Optional[int] = Field(
        default=None, description="Max side of overlay preview PNGs in lean profile; null = full resolution"
    )
    elapsed_ms: int
    num_candidates: int
    num_candidates_total: Optional[int] = Field(
        default=None,
        description="Candidates SAM 3 returned before the top-N max_candidates cut (>= num_candidates)",
    )
    candidates: List[CandidateMeta] = Field(default_factory=list)


class ErrorResponse(BaseModel):
    detail: str


In [ ]:
%%writefile /content/server_main.py
"""Colab server runner and helper utilities for AI Artwork Extractor.

Performs engine setup, app creation, background uvicorn server startup,
and Cloudflare Quick Tunnel exposure.
"""
from __future__ import annotations

import gc
import logging
import os
import re
import subprocess
import threading
import time
import urllib.request
from typing import Any, Optional, Tuple

logger = logging.getLogger("colab.server_main")


def build_sam_engine(
    device: str = "cuda",
    confidence_threshold: float = 0.35,
) -> Any:
    """Build and load Sam3Engine onto target device.

    Imports sam3 lazily so CPU/non-GPU test environments can import server_main.
    """
    from server_reference.sam3_engine import Sam3Engine

    engine = Sam3Engine(device=device, confidence_threshold=confidence_threshold)
    engine.load()
    return engine


def build_assistant(
    model_id: str = "Qwen/Qwen3-VL-4B-Instruct",
    enabled: bool = True,
    min_free_vram_gb: float = 5.0,
    device: str = "cuda",
) -> Any:
    """Build and optionally load QwenPromptAssistant in 4-bit mode.

    Cleans GPU cache before measuring VRAM and loading, so previous SAM inference
    does not cause spurious VRAM check failures.
    """
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    gc.collect()

    from server_reference.errors import PromptAssistantUnavailable
    from server_reference.qwen_prompt_assistant import QwenPromptAssistant

    assistant = QwenPromptAssistant(
        model_id=model_id,
        enabled=enabled,
        min_free_vram_gb=min_free_vram_gb,
        max_image_side=768,
        max_new_tokens=160,
        device=device,
    )
    if enabled:
        try:
            assistant.load()
        except PromptAssistantUnavailable as err:
            logger.warning("Prompt assistant unavailable: %s", err)
        except Exception as err:
            logger.warning("Unexpected error loading assistant: %s", err)
    return assistant


def start_uvicorn(
    app: Any,
    host: str = "127.0.0.1",
    port: int = 8000,
    timeout: float = 30.0,
) -> Tuple[Any, threading.Thread]:
    """Start uvicorn server in a daemon background thread and wait for /health response."""
    import requests
    import uvicorn

    config = uvicorn.Config(
        app=app,
        host=host,
        port=port,
        log_level="info",
        loop="asyncio",
    )
    server = uvicorn.Server(config=config)
    thread = threading.Thread(target=server.run, daemon=True, name="uvicorn-worker-thread")
    thread.start()

    health_url = f"http://{host}:{port}/health"
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            resp = requests.get(health_url, timeout=1.0)
            if resp.status_code == 200:
                logger.info("Local server answered at %s", health_url)
                return server, thread
        except Exception:
            pass
        time.sleep(0.5)

    raise RuntimeError(f"Timed out waiting for uvicorn server to start at {health_url}")


def parse_tunnel_url_from_text(text: str) -> Optional[str]:
    """Extract https://*.trycloudflare.com from log text."""
    pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    match = pattern.search(text)
    return match.group(0) if match else None


def start_cloudflared(
    port: int = 8000,
    binary_path: str = "/content/cloudflared",
    log_path: str = "/content/cloudflared.log",
    timeout: float = 60.0,
    existing_proc: Optional[subprocess.Popen] = None,
) -> Tuple[subprocess.Popen, str]:
    """Download (if needed) and run cloudflared Quick Tunnel, returning process and URL.

    Re-run safe: cleanly terminates any existing_proc and its log handle.
    Keeps a reference to the opened log file on success (attached to proc.log_file)
    and closes the log file on failure to avoid leaking file descriptors.
    """
    if existing_proc is not None:
        stop_server_and_tunnel(tunnel_proc=existing_proc)

    if not os.path.exists(binary_path):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        logger.info("Downloading cloudflared from %s to %s", url, binary_path)
        try:
            urllib.request.urlretrieve(url, binary_path)
        except Exception as err:
            raise RuntimeError(f"Failed to download cloudflared binary from {url}: {err}") from err
        try:
            os.chmod(binary_path, 0o755)
        except Exception as err:
            raise RuntimeError(f"Failed to set executable permissions on {binary_path}: {err}") from err

    log_file = open(log_path, "w", encoding="utf-8")
    proc: Optional[subprocess.Popen] = None
    try:
        proc = subprocess.Popen(
            [binary_path, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )

        start_time = time.time()
        tunnel_url = None

        while time.time() - start_time < timeout:
            if os.path.exists(log_path):
                with open(log_path, "r", errors="ignore", encoding="utf-8") as f:
                    content = f.read()
                    tunnel_url = parse_tunnel_url_from_text(content)
                    if tunnel_url:
                        break
            if proc.poll() is not None:
                raise RuntimeError(f"cloudflared exited unexpectedly with code {proc.returncode}")
            time.sleep(1.0)

        if not tunnel_url:
            tail = ""
            if os.path.exists(log_path):
                with open(log_path, "r", errors="ignore", encoding="utf-8") as f:
                    tail = f.read()[-500:]
            raise RuntimeError(f"Failed to find Cloudflare tunnel URL within {timeout}s:\n{tail}")

        # Attach log file handle to process so it can be closed upon shutdown
        setattr(proc, "log_file", log_file)
        return proc, tunnel_url

    except Exception:
        if proc is not None:
            try:
                proc.terminate()
                proc.wait(timeout=2)
            except Exception:
                try:
                    proc.kill()
                except Exception:
                    pass
        if not log_file.closed:
            try:
                log_file.close()
            except Exception:
                pass
        raise


def stop_server_and_tunnel(
    server: Optional[Any] = None,
    tunnel_proc: Optional[subprocess.Popen] = None,
    thread: Optional[threading.Thread] = None,
    timeout: float = 10.0,
) -> None:
    """Terminate the cloudflared process, close its log file, signal uvicorn server to exit,
    and wait for the uvicorn background thread to join so port 8000 is freed for restart.
    """
    if tunnel_proc is not None:
        try:
            tunnel_proc.terminate()
            tunnel_proc.wait(timeout=5)
        except Exception:
            try:
                tunnel_proc.kill()
            except Exception:
                pass
        finally:
            log_file = getattr(tunnel_proc, "log_file", None)
            if log_file and not log_file.closed:
                try:
                    log_file.close()
                except Exception:
                    pass

    if server is not None:
        server.should_exit = True

    if thread is not None and thread.is_alive():
        thread.join(timeout=timeout)

    # Brief pause to ensure OS releases bound sockets
    if server is not None or thread is not None:
        time.sleep(0.5)


In [ ]:
import sys
# append (không insert(0)) để /content không che các package đã cài
if "/content" not in sys.path:
    sys.path.append("/content")
print("server_reference and server_main loaded into sys.path.")


## 6. Nạp SAM 3 Image Model và Processor (Chỉ nạp 1 lần)

Khởi tạo `Sam3Engine` với `SAM_DEVICE` và `DEFAULT_THRESHOLD` từ Mục 1b. Thời gian nạp và dung lượng VRAM sử dụng sẽ được hiển thị.

In [ ]:
import gc, time, torch
from server_reference.sam3_engine import Sam3Engine

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("Loading SAM 3 image model and processor...")
start_time = time.time()
sam_engine = Sam3Engine(device=SAM_DEVICE, confidence_threshold=DEFAULT_THRESHOLD)
sam_engine.load()
load_elapsed = time.time() - start_time

alloc_gb = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
print(f"SAM 3 loaded successfully in {load_elapsed:.2f} seconds.")
print(f"GPU memory allocated: {alloc_gb:.2f} GB")


## 7. SAM 3 Smoke Test

Chạy thử nghiệm một inference text mode đơn giản trên ảnh mẫu nhân tạo để đảm bảo engine hoạt động bình thường. Nếu trả về 0 candidates, notebook vẫn xử lý an toàn không gây crash.

In [ ]:
import time
from PIL import Image, ImageDraw
from server_reference.schemas import Mode

# Tạo ảnh mẫu nhân tạo có patch màu rõ rệt để smoke test
smoke_img = Image.new("RGB", (600, 600), color=(245, 245, 245))
draw = ImageDraw.Draw(smoke_img)
draw.rectangle([150, 150, 450, 450], fill=(220, 50, 50), outline=(20, 20, 20), width=4)
draw.text((230, 290), "ARTWORK", fill=(255, 255, 255))
smoke_img_path = "/content/smoke_sample.png"
smoke_img.save(smoke_img_path)

print(f"Running SAM 3 smoke test on test image ({smoke_img.size})...")
t0 = time.time()
smoke_prompt = "red square patch"
try:
    smoke_result = sam_engine.segment(
        image=smoke_img,
        mode=Mode.TEXT,
        prompt=smoke_prompt,
        bbox_cxcywh_norm=None,
        threshold=0.20,
    )
    smoke_elapsed = time.time() - t0
    num_cands = len(smoke_result.scores)
    print(f"SAM 3 smoke test completed in {smoke_elapsed:.2f}s: {num_cands} candidate(s) found.")
    if num_cands > 0:
        print(f"Candidate scores: {smoke_result.scores.tolist()}")
    else:
        print("Note: 0 candidates found for synthetic image. Handled safely without crash.")
except Exception as e:
    print(f"SAM 3 smoke test error: {e}")


## 8. Cấu hình và nạp Qwen3-VL-4B Prompt Assistant (Tùy chọn, 4-bit NF4)

Dọn dẹp cache VRAM (`gc.collect(); torch.cuda.empty_cache()`) để tránh việc bộ nhớ cache của bước smoke test làm sai lệch bước kiểm tra VRAM trống. Gọi `server_main.build_assistant(...)` để nạp model. Nếu VRAM còn lại dưới `QWEN_MIN_FREE_VRAM_GB` hoặc nạp thất bại, Prompt Assistant sẽ tự vô hiệu hóa và SAM 3 manual mode vẫn hoạt động bình thường.

In [ ]:
import gc, torch
import server_main
from server_reference.errors import PromptAssistantUnavailable

# Dọn dẹp cache GPU trước khi đo VRAM và nạp Qwen
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"VRAM before Qwen load: {free_bytes / 1024**3:.2f} GB free / {total_bytes / 1024**3:.2f} GB total")

if ENABLE_QWEN:
    print("Attempting to load Qwen3-VL-4B Prompt Assistant via server_main.build_assistant...")
    try:
        assistant = server_main.build_assistant(
            model_id="Qwen/Qwen3-VL-4B-Instruct",
            enabled=ENABLE_QWEN,
            min_free_vram_gb=QWEN_MIN_FREE_VRAM_GB,
            device="cuda",
        )
        if assistant.loaded:
            print("Qwen Prompt Assistant loaded successfully!")
        else:
            print(f"Prompt Assistant disabled — SAM 3 manual mode still works (Reason: {assistant.last_error or 'insufficient VRAM or disabled'})")
    except PromptAssistantUnavailable as e:
        print(f"Prompt Assistant disabled — SAM 3 manual mode still works (Reason: {e})")
        assistant = server_main.build_assistant(enabled=False)
    except Exception as e:
        print(f"Prompt Assistant disabled — SAM 3 manual mode still works (Unexpected error: {e})")
        assistant = server_main.build_assistant(enabled=False)
else:
    print("Qwen Prompt Assistant disabled by user (ENABLE_QWEN = False).")
    assistant = server_main.build_assistant(enabled=False)

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"VRAM after Qwen load attempt: {free_bytes / 1024**3:.2f} GB free / {total_bytes / 1024**3:.2f} GB total")

print("VRAM Report:", assistant.vram_report())


## 9. Qwen Prompt Assistant Smoke Test (Tiếng Việt)

Gửi yêu cầu tiếng Việt kèm ảnh mẫu để kiểm tra khả năng đề xuất prompt có cấu trúc (bỏ qua nếu Qwen không khả dụng).

In [ ]:
from server_reference.schemas import ScopePreference

if assistant.available and assistant.loaded:
    print("Running Qwen smoke test with Vietnamese request...")
    vietnamese_request = "Lấy miếng patch thêu hình vuông ở ngực áo, bao gồm cả viền vải."
    try:
        suggestion = assistant.suggest(
            image=smoke_img,
            user_request=vietnamese_request,
            scope_preference=ScopePreference.AUTO,
        )
        print("Qwen Suggestion Result:")
        print(f"  Target: {suggestion.target_description}")
        print(f"  Scope: {suggestion.scope}")
        print(f"  SAM Prompt: {suggestion.sam_prompt}")
        print(f"  Alternatives: {suggestion.alternative_prompts}")
        print(f"  Needs clarification: {suggestion.needs_clarification}")
    except Exception as e:
        print(f"Qwen smoke test error: {e}")
else:
    print("Skipping Qwen smoke test: Prompt Assistant is not available or not loaded.")


## 10. Cấu hình bảo mật và tham số Server

Tạo Bearer Token ngẫu nhiên (hoặc đọc từ Colab Secret `ARTWORK_API_TOKEN`). `MAX_UPLOAD_MB` và `DEFAULT_THRESHOLD` được lấy từ cấu hình chung ở Mục 1b.

In [ ]:
import secrets

API_TOKEN = None
try:
    from google.colab import userdata
    API_TOKEN = userdata.get("ARTWORK_API_TOKEN")
except Exception:
    API_TOKEN = None

if not API_TOKEN or len(API_TOKEN.strip()) < 16:
    API_TOKEN = secrets.token_urlsafe(32)

print("Server configuration ready:")
print(f"  MAX_UPLOAD_MB    : {MAX_UPLOAD_MB}")
print(f"  DEFAULT_THRESHOLD: {DEFAULT_THRESHOLD}")
print(f"  API_TOKEN length : {len(API_TOKEN)} chars (will be displayed with tunnel URL)")


## 11. Khởi chạy FastAPI & Uvicorn (Daemon Background Thread)

Khởi tạo FastAPI app và gọi `server_main.start_uvicorn(...)` để chạy uvicorn trong background thread. Tự động dừng server cũ và đợi giải phóng port 8000 nếu cell được chạy lại.

In [ ]:
from server_reference.api import create_app
import server_main

# Re-run safe: dừng server trước đó nếu có và chờ thread kết thúc để giải phóng port 8000
if "server" in globals() and server:
    server_main.stop_server_and_tunnel(
        server=server,
        thread=globals().get("server_thread"),
    )

app = create_app(
    sam_engine=sam_engine,
    assistant=assistant,
    api_token=API_TOKEN,
    max_upload_mb=MAX_UPLOAD_MB,
    default_threshold=DEFAULT_THRESHOLD,
    max_candidates=MAX_CANDIDATES,
    cors_origins=CORS_ORIGINS,
    overlay_max_side=OVERLAY_MAX_SIDE,
)

print("Starting Uvicorn server in background thread via server_main.start_uvicorn...")
server, server_thread = server_main.start_uvicorn(
    app=app,
    host="127.0.0.1",
    port=8000,
    timeout=30.0,
)
print("Local Uvicorn server is online on http://127.0.0.1:8000/health")


## 12. Cài đặt và khởi chạy Cloudflare Quick Tunnel

Gọi `server_main.start_cloudflared(...)` để tải binary (nếu chưa có), cấp quyền thực thi và mở tunnel. Tự động dừng tunnel cũ nếu cell được chạy lại và trích xuất public URL `https://*.trycloudflare.com`.

In [ ]:
import server_main

# Re-run safe: truyền tiến trình tunnel cũ để server_main dọn dẹp sạch sẽ
existing_tunnel = globals().get("tunnel_proc")

print("Starting Cloudflare Quick Tunnel via server_main.start_cloudflared...")
tunnel_proc, tunnel_url = server_main.start_cloudflared(
    port=8000,
    binary_path="/content/cloudflared",
    log_path="/content/cloudflared.log",
    timeout=60.0,
    existing_proc=existing_tunnel,
)

print(f"Cloudflare Tunnel online: {tunnel_url}")


## 13. Thông tin kết nối & Cấu hình cho Web UI

Sao chép thông tin URL và Bearer Token để dán vào hộp thoại kết nối trên Web UI ('Cài đặt kết nối API').

In [ ]:
print("=" * 72)
print("  AI ARTWORK EXTRACTOR — INFERENCE SERVER ONLINE")
print("=" * 72)
print(f"Public API Base URL: {tunnel_url}")
print(f"Bearer Token       : {API_TOKEN}")
print("WARNING: Do NOT share this token. It grants full access to your GPU server.")
print("=" * 72)

print("\n# Dán URL và Token vào hộp thoại kết nối trên Web UI ('Cài đặt kết nối API'):")
print(f"COLAB_API_URL={tunnel_url}")
print(f"COLAB_API_TOKEN={API_TOKEN}")
print("-" * 72)

print("\n# Test curl commands:")
print(f"# 1. Health check (no auth):")
print(f"curl -s {tunnel_url}/health\n")

print(f"# 2. Segmentation request (text mode):")
print(f'curl -X POST "{tunnel_url}/api/v1/segment" \\')
print(f'     -H "Authorization: Bearer {API_TOKEN}" \\')
print(f'     -F "image=@sample.jpg" \\')
print(f'     -F "mode=text" \\')
print(f'     -F "prompt=square embroidered fabric patch" \\')
print(f'     --output result.zip')
print("=" * 72)


## 14. Kiểm tra Endpoint qua Public Tunnel

Gửi request `/health` và request `/api/v1/segment` thử nghiệm qua đường tunnel công khai.

In [ ]:
import os, requests

print(f"Testing public health endpoint through tunnel: {tunnel_url}/health")
try:
    resp = requests.get(f"{tunnel_url}/health", timeout=15)
    print(f"Health response (status {resp.status_code}):\n{resp.json()}")
except Exception as e:
    print(f"Tunnel health check failed: {e}")

if os.path.exists("/content/smoke_sample.png"):
    print("\nTesting /api/v1/segment via public tunnel...")
    try:
        with open("/content/smoke_sample.png", "rb") as f:
            seg_resp = requests.post(
                f"{tunnel_url}/api/v1/segment",
                headers={"Authorization": f"Bearer {API_TOKEN}"},
                files={"image": ("smoke_sample.png", f, "image/png")},
                data={"mode": "text", "prompt": "red square patch", "confidence_threshold": "0.20"},
                timeout=60,
            )
        print(f"Segmentation test status: {seg_resp.status_code}")
        if seg_resp.status_code == 200:
            print(f"Received result.zip ({len(seg_resp.content)} bytes)")
        else:
            print(f"Error detail: {seg_resp.text}")
    except Exception as e:
        print(f"Tunnel segmentation check failed: {e}")


## 15. Dừng Server và Cloudflare Tunnel

CẢNH BÁO: Không tự động chạy cell này khi dùng 'Run all'! Để tránh server bị dừng sau khi chạy toàn bộ notebook, cell này được bảo vệ bởi cờ `STOP_SERVER = False`. Khi muốn dừng server Uvicorn và đóng Cloudflare Tunnel, đổi `STOP_SERVER = True` trong cell dưới đây rồi chạy lại cell này.

In [ ]:
import server_main

STOP_SERVER = False  # đổi thành True rồi chạy cell này để dừng

if STOP_SERVER:
    print("Shutting down server and tunnel via server_main.stop_server_and_tunnel...")
    server_main.stop_server_and_tunnel(
        server=globals().get("server"),
        tunnel_proc=globals().get("tunnel_proc"),
        thread=globals().get("server_thread"),
    )
    print("Shutdown complete.")
else:
    print("Bỏ qua dừng server (STOP_SERVER = False). Server vẫn đang chạy bình thường.")
    print("Khi muốn dừng hẳn server, hãy sửa `STOP_SERVER = True` rồi chạy lại cell này.")


## 16. Trạng thái server

Kiểm tra và xác nhận server vẫn đang hoạt động sau khi thực hiện Run all.

In [ ]:
import requests

tunnel_url = globals().get("tunnel_url")
print("=" * 72)
print("  TRẠNG THÁI SERVER — INFERENCE SERVER ĐANG HOẠT ĐỘNG")
print("=" * 72)
print(f"Public API Base URL: {tunnel_url}")

if tunnel_url:
    try:
        resp = requests.get(f"{tunnel_url}/health", timeout=10)
        print(f"Health Check (Status {resp.status_code}):")
        print(f"  {resp.json()}")
    except Exception as e:
        print(f"Health Check Error: {e}")
else:
    print("Chưa có tunnel_url. Hãy kiểm tra các cell khởi chạy phía trên.")
print("=" * 72)


## Hướng dẫn xử lý sự cố (Troubleshooting)

- **Tunnel URL thay đổi khi khởi động lại**:
  Cloudflare Quick Tunnel tự động cấp một URL `https://*.trycloudflare.com` ngẫu nhiên mỗi khi tiến trình chạy lại.   Khi restart tunnel, bạn chỉ cần cập nhật URL mới vào hộp thoại kết nối trên Web UI ('Cài đặt kết nối API').

- **Quick Tunnel aborts responses > 100s (Quá thời gian chờ phản hồi)**:
  Cloudflare Quick Tunnel tự động ngắt kết nối với các response kéo dài hơn ~100 giây.   Đối với hình ảnh rất lớn (ví dụ 2000×1805 trở lên), hãy giảm `MAX_CANDIDATES` ở Mục 1b (mặc định 12)   hoặc sử dụng chế độ Box / Text+Box mode để khoanh vùng thay vì để SAM tìm kiếm trên toàn bộ ảnh.

- **Prompt phải là tiếng Anh (English)**:
  Mô hình Meta SAM 3 chỉ hiểu tiếng Anh (các ký tự ASCII). Vì vậy `sam_prompt` trong request phải luôn là   cụm danh từ tiếng Anh (ví dụ 'embroidered red poppy flowers' thay vì 'bông hoa đỏ'). Qwen Prompt Assistant   sẽ tự động chuyển đổi yêu cầu tiếng Việt của designer sang tiếng Anh.

- **`ModuleNotFoundError: No module named 'sam3.model_builder'`**:
  Xảy ra nếu repo SAM 3 được clone vào `/content/sam3` (trùng tên package) bởi notebook phiên bản cũ. Chọn **Runtime → Disconnect and delete runtime** rồi Run all lại notebook này (clone vào `/content/sam3_repo`).

- **CUDA Out-of-Memory (OOM) hoặc muốn tắt Qwen**:
  GPU T4 có 16GB VRAM. Nếu việc nạp Qwen3-VL-4B cùng SAM 3 gây quá tải bộ nhớ:
  1. Vào menu **Runtime → Restart session**.
  2. Ở **Mục 1b (Cell cấu hình chung)** ở đầu notebook, đặt `ENABLE_QWEN = False`.
  3. Chạy lần lượt các cell từ trên xuống dưới. Các thư viện nặng của Qwen (bitsandbytes, accelerate) sẽ không bị tải/cài và SAM 3 sẽ chạy độc lập thoải mái trong VRAM.

- **Colab Runtime bị ngắt kết nối (Disconnect / Timeout)**:
  Google Colab có thể ngắt kết nối khi session không có tương tác trong một thời gian dài.
  - Kết nối lại GPU T4 runtime.
  - Chạy lại các cell trong notebook.
  - Cập nhật URL và Token mới vào ứng dụng UI.

- **Lỗi xác thực 401 Unauthorized**:
  Đảm bảo header `Authorization: Bearer <API_TOKEN>` được đính kèm trong mọi request. Token được in ra ở Mục 13.
